In [1]:
import polars as pl
import math
from pathlib import Path

pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_width_chars(250)

polars.config.Config

In [2]:
GTFS = Path("raw/processed_gtfs")

routes = pl.read_parquet(GTFS / "routes.parquet")
trips = pl.read_parquet(GTFS / "trips.parquet")
stop_times = pl.read_parquet(GTFS / "stop_times.parquet")
stops = pl.read_parquet(GTFS / "stops.parquet")
shapes = pl.read_parquet(GTFS / "shapes.parquet")

In [3]:
# # ============================================================
# # Clean stop coordinates
# # ============================================================

# stops = stops.with_columns(
#     pl.col("stop_lat")
#         .str.strip_chars()
#         .cast(pl.Float64),

#     pl.col("stop_lon")
#         .str.strip_chars()
#         .cast(pl.Float64)
# )

In [4]:
# ============================================================
# GTFS time parser
# Handles times like 25:13:00
# ============================================================

def gtfs_to_seconds(t):

    if t is None:
        return None

    h, m, s = map(int, t.split(":"))

    return h * 3600 + m * 60 + s

In [5]:
# ============================================================
# Convert schedule times
# ============================================================

stop_times = stop_times.with_columns(

    pl.col("arrival_time")
    .map_elements(
        gtfs_to_seconds,
        return_dtype=pl.Int64
    )
    .alias("arrival_seconds"),

    pl.col("departure_time")
    .map_elements(
        gtfs_to_seconds,
        return_dtype=pl.Int64
    )
    .alias("departure_seconds")

)

In [6]:
# ============================================================
# Sort tables
# ============================================================

stop_times = stop_times.sort(
    ["trip_id", "stop_sequence"]
)

shapes = shapes.sort(
    ["shape_id", "shape_pt_sequence"]
)

In [7]:
# ============================================================
# Previous shape point
# ============================================================

shapes = shapes.with_columns(

    pl.col("shape_pt_lat")
    .shift()
    .over("shape_id")
    .alias("prev_lat"),

    pl.col("shape_pt_lon")
    .shift()
    .over("shape_id")
    .alias("prev_lon")

)

In [8]:
# ============================================================
# Compute edge length (Pure Polars)
# ============================================================

R = 6371000.0

lat1 = pl.col("prev_lat") * math.pi / 180
lon1 = pl.col("prev_lon") * math.pi / 180

lat2 = pl.col("shape_pt_lat") * math.pi / 180
lon2 = pl.col("shape_pt_lon") * math.pi / 180

dlat = lat2 - lat1
dlon = lon2 - lon1

a = (
    (dlat / 2).sin().pow(2)
    +
    lat1.cos()
    *
    lat2.cos()
    *
    (dlon / 2).sin().pow(2)
)

c = 2 * pl.arctan2(
    a.sqrt(),
    (1 - a).sqrt()
)

shapes = shapes.with_columns(

    pl.when(
        pl.col("prev_lat").is_null()
    )

    .then(0.0)

    .otherwise(
        R * c
    )

    .alias("edge_length")

)

In [9]:
# ============================================================
# Cumulative distance along every shape
# ============================================================

shapes = shapes.with_columns(

    pl.col("edge_length")
      .cum_sum()
      .over("shape_id")
      .alias("shape_distance")

)

In [10]:
# ============================================================
# Create stop -> shape lookup
# ============================================================

stop_shape = (

    stop_times

    .join(

        trips.select([
            "trip_id",
            "shape_id"
        ]),

        on="trip_id"

    )

    .join(

        stops.select([
            "stop_id",
            "stop_lat",
            "stop_lon"
        ]),

        on="stop_id"

    )

)

In [11]:
# ============================================================
# Join all shape points
# ============================================================

stop_shape = stop_shape.join(

    shapes.select([
        "shape_id",
        "shape_pt_sequence",
        "shape_pt_lat",
        "shape_pt_lon",
        "shape_distance"
    ]),

    on="shape_id"

)

In [12]:
# ============================================================
# Distance stop -> shape point
# Pure Polars
# ============================================================

lat1 = pl.col("stop_lat") * math.pi / 180
lon1 = pl.col("stop_lon") * math.pi / 180

lat2 = pl.col("shape_pt_lat") * math.pi / 180
lon2 = pl.col("shape_pt_lon") * math.pi / 180

dlat = lat2 - lat1
dlon = lon2 - lon1

a = (
    (dlat / 2).sin().pow(2)
    +
    lat1.cos()
    *
    lat2.cos()
    *
    (dlon / 2).sin().pow(2)
)

c = 2 * pl.arctan2(
    a.sqrt(),
    (1 - a).sqrt()
)

stop_shape = stop_shape.with_columns(

    (R * c)

    .alias("distance_to_shape")

)

In [13]:
# ============================================================
# Keep nearest shape point
# ============================================================

stop_shape_mapping = (

    stop_shape

    .sort("distance_to_shape")

    .group_by([
        "trip_id",
        "stop_id"
    ])

    .first()

    .select([

        "trip_id",

        "shape_id",

        "stop_id",

        "shape_pt_sequence",

        "shape_distance"

    ])

)

: 

In [14]:
print(stop_shape_mapping.shape)

stop_shape_mapping.head()

(259787, 5)


trip_id,shape_id,stop_id,shape_pt_sequence,shape_distance
str,str,i64,i64,f64
"""MV_C6-Weekday-116100_M4_439""","""M040923""",400633,730002,14385.525885
"""OF_C6-Saturday-060200_M1_117""","""M010114""",400042,400005,8870.041856
"""MV_C6-Weekday-132000_M3_341""","""M020238""",404837,630004,12466.310664
"""OF_C6-Weekday-047600_M1_109""","""M010114""",400030,290005,6561.480778
"""MV_C6-Sunday-073400_M4_426""","""M040918""",400628,680002,13597.212745


In [15]:
(
    stop_shape_mapping
    .filter(pl.col("trip_id") == "FB_C6-Saturday-066200_B41_230")
    .sort("shape_distance")
    .select([
        "stop_id",
        "shape_distance"
    ])
)

stop_id,shape_distance
i64,f64


In [16]:
(
    stop_shape_mapping
    .filter(pl.col("trip_id") == "FB_C6-Saturday-066200_B41_230")
    .sort("shape_distance")
    .with_columns(
        (
            pl.col("shape_distance")
            - pl.col("shape_distance").shift(1)
        ).alias("segment_length")
    )
)

trip_id,shape_id,stop_id,shape_pt_sequence,shape_distance,segment_length
str,str,i64,i64,f64,f64


In [18]:
stop_shape_mapping.write_parquet("processed/stop_shape_mapping.parquet")

In [19]:
segments = (
    stop_times
    .with_columns([

        pl.col("stop_id")
        .shift(-1)
        .over("trip_id")
        .alias("next_stop_id"),

        pl.col("stop_sequence")
        .shift(-1)
        .over("trip_id")
        .alias("next_stop_sequence"),

        pl.col("arrival_seconds")
        .shift(-1)
        .over("trip_id")
        .alias("next_arrival"),

        pl.col("departure_seconds")
        .shift(-1)
        .over("trip_id")
        .alias("next_departure")

    ])
    .filter(
        pl.col("next_stop_id").is_not_null()
    )
)

In [20]:
segments = segments.with_columns(

    (
        pl.col("next_arrival")
        -
        pl.col("departure_seconds")
    )

    .alias("scheduled_travel_time")

)

In [21]:
segments = segments.join(

    trips.select([
        "trip_id",
        "route_id",
        "direction_id",
        "shape_id"
    ]),

    on="trip_id",
    how="left"

)

In [22]:
segments = segments.join(

    stops.select([
        "stop_id",
        "stop_name",
        "stop_lat",
        "stop_lon"
    ]),

    on="stop_id",
    how="left"

)

In [23]:
segments = segments.rename({

    "stop_name": "start_stop_name",
    "stop_lat": "start_lat",
    "stop_lon": "start_lon"

})

In [24]:
segments = segments.join(

    stops.select([
        "stop_id",
        "stop_name",
        "stop_lat",
        "stop_lon"
    ]),

    left_on="next_stop_id",
    right_on="stop_id",
    how="left"

)

In [25]:
segments = segments.rename({

    "stop_name": "end_stop_name",
    "stop_lat": "end_lat",
    "stop_lon": "end_lon"

})

In [26]:
segments = (
    segments
    .join(
        stop_shape_mapping.select([
            "trip_id",
            "stop_id",
            "shape_distance"
        ]),
        on=["trip_id", "stop_id"],
        how="left"
    )
)

In [27]:
segments = (
    segments
    .join(
        stop_shape_mapping.select([
            "trip_id",
            "stop_id",
            "shape_distance"
        ]).rename({
            "stop_id": "next_stop_id",
            "shape_distance": "next_shape_distance"
        }),
        on=["trip_id", "next_stop_id"],
        how="left"
    )
)

In [28]:
segments = segments.with_columns(

    (
        pl.col("next_shape_distance")
        -
        pl.col("shape_distance")
    )
    .abs()
    .alias("segment_length")

)

In [29]:
segments = segments.with_columns(

    pl.concat_str([

        pl.col("route_id"),

        pl.col("direction_id").cast(pl.String),

        pl.col("shape_id"),

        pl.col("stop_id").cast(pl.String),

        pl.col("next_stop_id").cast(pl.String)

    ], separator="_").alias("segment_id")

)

In [30]:
segment_network = (

    segments

    .select([

        "segment_id",

        "route_id",
        "direction_id",
        "shape_id",

        "stop_id",
        "next_stop_id",

        "stop_sequence",
        "next_stop_sequence",

        "start_stop_name",
        "end_stop_name",

        "start_lat",
        "start_lon",

        "end_lat",
        "end_lon",

        "shape_distance",
        "next_shape_distance",

        "segment_length",

        "scheduled_travel_time"

    ])

    .unique(subset=["segment_id"])

    .sort([
        "route_id",
        "direction_id",
        "shape_distance"
    ])

)

In [31]:
segment_schedule = (

    segments

    .select([

        "trip_id",

        "route_id",
        "direction_id",

        "segment_id",

        "departure_seconds",

        "next_arrival",

        "scheduled_travel_time"

    ])

    .sort([
        "trip_id",
        "departure_seconds"
    ])

)

In [32]:
OUTPUT = Path("processed")

OUTPUT.mkdir(exist_ok=True)

segment_network.write_parquet(
    OUTPUT / "segment_network.parquet"
)

segment_schedule.write_parquet(
    OUTPUT / "segment_schedule.parquet"
)

In [33]:
segment_network.head()

segment_id,route_id,direction_id,shape_id,stop_id,next_stop_id,stop_sequence,next_stop_sequence,start_stop_name,end_stop_name,start_lat,start_lon,end_lat,end_lon,shape_distance,next_shape_distance,segment_length,scheduled_travel_time
str,str,i64,str,i64,i64,i64,i64,str,str,f64,f64,f64,f64,f64,f64,f64,i64
"""M1_0_M010110_400080_400081""","""M1""",0,"""M010110""",400080,400081,1,2,"""CENTRE ST/BROOME ST""","""CLEVELAND PL/SPRING ST""",40.720678,-73.997674,40.722433,-73.997008,0.0,202.383616,202.383616,63
"""M1_0_M010114_400080_400081""","""M1""",0,"""M010114""",400080,400081,1,2,"""CENTRE ST/BROOME ST""","""CLEVELAND PL/SPRING ST""",40.720678,-73.997674,40.722433,-73.997008,0.0,202.383616,202.383616,45
"""M1_0_M010113_405527_400003""","""M1""",0,"""M010113""",405527,400003,1,2,"""4 AV/E 10 ST""","""4 AV/E 13 ST""",40.731354,-73.990292,40.733936,-73.98972,0.0,289.976163,289.976163,136
"""M1_0_M010114_400081_405523""","""M1""",0,"""M010114""",400081,405523,2,3,"""CLEVELAND PL/SPRING ST""","""LAFAYETTE ST/EAST HOUSTON ST""",40.722433,-73.997008,40.725154,-73.995195,202.383616,545.821317,343.437701,76
"""M1_0_M010110_400081_405523""","""M1""",0,"""M010110""",400081,405523,2,3,"""CLEVELAND PL/SPRING ST""","""LAFAYETTE ST/EAST HOUSTON ST""",40.722433,-73.997008,40.725154,-73.995195,202.383616,545.821317,343.437701,106


In [34]:
segment_schedule.head()

trip_id,route_id,direction_id,segment_id,departure_seconds,next_arrival,scheduled_travel_time
str,str,i64,str,i64,i64,i64
"""MV_C6-Saturday-004600_M2_201""","""M2""",0,"""M2_0_M020100_400241_400242""",2760,2792,32
"""MV_C6-Saturday-004600_M2_201""","""M2""",0,"""M2_0_M020100_400242_404129""",2792,2820,28
"""MV_C6-Saturday-004600_M2_201""","""M2""",0,"""M2_0_M020100_404129_403338""",2820,2854,34
"""MV_C6-Saturday-004600_M2_201""","""M2""",0,"""M2_0_M020100_403338_404837""",2854,2877,23
"""MV_C6-Saturday-004600_M2_201""","""M2""",0,"""M2_0_M020100_404837_400248""",2877,3032,155


In [35]:
shape_points = (
    shapes
    .select([
        "shape_id",
        "shape_pt_sequence",
        "shape_pt_lat",
        "shape_pt_lon",
        "shape_distance"
    ])
    .sort([
        "shape_id",
        "shape_pt_sequence"
    ])
)

shape_points.write_parquet(
    "processed/shape_points.parquet"
)

In [36]:
trip_shapes = (
    trips
    .select([
        "trip_id",
        "route_id",
        "direction_id",
        "shape_id"
    ])
)

trip_shapes.write_parquet(
    "processed/trip_shapes.parquet"
)